# 03 · Logit ve Probit: Sınavı Geçme Olasılığı

Bağımlı değişken **0/1** olduğunda (geçti/kaldı, işe girdi/girmedi, kredi ödendi/ödenmedi) doğrusal regresyon 0–1 aralığının dışına çıkan olasılıklar üretebilir. Logit ve probit modelleri olasılığı her zaman 0 ile 1 arasında tutar.

**Öğreneceklerin**
1. Logit ve probit modellerini tahmin etmek
2. Katsayıları değil **ortalama marjinal etkileri (AME)** yorumlamak
3. Odds oranlarını ve sınıflandırma tablosunu okumak
4. Sonuçları tez formatında raporlamak

> ⚠️ Kullanılan veri **simüle edilmiştir**.

In [1]:
import ekonometri as ek

veri = ek.veri.yukle("sinav")
veri["gecti"].mean()

np.float64(0.8375)

## 1. Logit modeli

In [2]:
logit = ek.logit("gecti ~ calisma_saati + devam + anne_egitim + burs", veri, birimler={"calisma_saati": "saat", "devam": "puan"})
logit

Değişken,Katsayı,Std. hata,z,p,
Sabit,"-6,4756","0,8082","-8,01","<0,0001",***
burs: Evet,"0,7112","0,2691","2,64","0,0082",***
calisma_saati,"0,3245","0,0368","8,83","<0,0001",***
devam,"0,0602","0,0078","7,75","<0,0001",***
anne_egitim,"0,0929","0,0302","3,08","0,0021",***


Tablonun altındaki uyarıya dikkat: logit katsayıları **olasılıktaki değişim değildir**, yalnızca yönü gösterir. Olasılık yorumu için marjinal etkilere bakılır:

In [3]:
logit.marjinal_etkiler()[['AME', 'std. hata', 'p']]

,AME,std. hata,p
calisma_saati,0.033342,0.003219,3.886638e-25
devam,0.006183,0.000681,1.047462e-19
anne_egitim,0.009547,0.003045,1.719573e-03
burs: Evet,0.068403,0.023715,3.922159e-03


In [4]:
logit.yorumla()

📊 Yorum — Logit: 'gecti' = 1 olma olasılığı

▸ Model geneli
  • Model, yalnızca sabit terimden oluşan modele göre anlamlı biçimde daha iyi (LR χ²(4) = 184,70, p < 0,0001).
  • McFadden sözde R² = 0,260 (EKK'deki R² gibi 'açıklanan varyans' olarak yorumlanmaz; modeller arası karşılaştırma içindir).
  • Gözlemlerin %86,4 kadarı doğru sınıflandırılıyor (eşik 0,5); örneklemde 'gecti' = 1 oranı %83,8.
  • Gözlem sayısı: 800 · standart hatalar: klasik (en çok olabilirlik, bilgi matrisi).

▸ Katsayılar (diğer değişkenler sabitken)
  • 'calisma_saati' 1 saat artınca 'gecti' = 1 olma olasılığı ortalama 3,33 yüzde puan artıyor — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'devam' 1 puan artınca 'gecti' = 1 olma olasılığı ortalama 0,62 yüzde puan artıyor — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'anne_egitim' 1 birim artınca 'gecti' = 1 olma olasılığı ortalama 0,95 yüzde puan artıyor — %1 düzeyinde istatistiksel olarak anlamlı (p = 0,0017).
  • 'burs' değişkeninde 'Evet' grubunun 'gecti' = 1 olma olasılığı, referans gruba ('Hayır') göre ortalama 6,84 yüzde puan daha yüksek — %1 düzeyinde istatistiksel olarak anlamlı (p = 0,0039).

▸ Notlar
  • Yorumlar ortalama marjinal etkilere (AME) dayanır: katsayıların kendisi olasılıktaki değişimi vermez, yalnızca yönü gösterir.
  • Marjinal etki her gözlem için farklıdır; burada tüm gözlemlerin ortalaması verildi (tablo: model.marjinal_etkiler()).
  • Odds (bahis) oranları için: model.odds_oranlari()
  • Bu yorumlar ilişkiyi anlatır; nedensellik için araştırma tasarımının buna izin vermesi gerekir.

## 2. Odds oranları ve sınıflandırma

In [5]:
logit.odds_oranlari()

,odds oranı,alt,üst
burs[T.Evet],2.036428,1.201749,3.450835
calisma_saati,1.383277,1.287145,1.486589
devam,1.062013,1.045966,1.078307
anne_egitim,1.097350,1.034295,1.164248


In [6]:
logit.siniflandirma()

,Tahmin: 0,Tahmin: 1
Gerçek: 0,38,92
Gerçek: 1,17,653


Doğru sınıflandırma oranı yüksek görünse de örneklemin büyük kısmı zaten sınavı geçmiş: **taban oran** ile karşılaştırmadan bu sayıya güvenme. Özgüllük (kalanları doğru bilme oranı) çok daha düşük.

## 3. Logit, probit ve doğrusal olasılık modeli yan yana

Katsayılar ölçek farkı nedeniyle farklıdır ama marjinal etkiler birbirine çok yakındır:

In [7]:
formul = "gecti ~ calisma_saati + devam + anne_egitim + burs"
probit = ek.probit(formul, veri)
dom = ek.ekk(formul, veri, kovaryans="HC1")   # doğrusal olasılık modeli
ek.karsilastir(dom, logit, probit, adlar=["DOM (EKK)", "Logit", "Probit"])

Değişken,DOM (EKK),Logit,Probit
burs: Evet,"0,0692***","0,7112***","0,4044***"
,"(0,0245)","(0,2691)","(0,1488)"
calisma_saati,"0,0221***","0,3245***","0,1725***"
,"(0,0021)","(0,0368)","(0,0188)"
devam,"0,0068***","0,0602***","0,0333***"
,"(0,000965)","(0,0078)","(0,0043)"
anne_egitim,"0,0079***","0,0929***","0,0522***"
,"(0,0030)","(0,0302)","(0,0169)"
Sabit,"-0,0148","-6,4756***","-3,5186***"
,"(0,0921)","(0,8082)","(0,4444)"


In [8]:
import pandas as pd
pd.DataFrame({"Logit AME": logit.marjinal_etkiler()["AME"], "Probit AME": probit.marjinal_etkiler()["AME"]}).round(4)

,Logit AME,Probit AME
calisma_saati,0.0333,0.0317
devam,0.0062,0.0061
anne_egitim,0.0095,0.0096
burs: Evet,0.0684,0.0699


## 4. Tez raporu

In [9]:
print(logit.rapor(etiketler={"calisma_saati": "Haftalık çalışma süresi", "gecti": "Geçme"}).paragraflar[4][:900])

Açıklayıcı değişkenlerin Geçme = 1 olma olasılığı üzerindeki ortalama marjinal etkileri Tablo 4 ile sunulmuştur. Diğer değişkenler sabitken, Haftalık çalışma süresi değişkeni 1 saat arttığında Geçme = 1 olma olasılığı ortalama 3,33 yüzde puan artmaktadır; bu etki %1 düzeyinde istatistiksel olarak anlamlıdır (AME = 0,0333; SH = 0,0032; z = 10,36; p < 0,001). Diğer değişkenler sabitken, 'devam' değişkeni 1 puan arttığında Geçme = 1 olma olasılığı ortalama 0,62 yüzde puan artmaktadır; bu etki %1 düzeyinde istatistiksel olarak anlamlıdır (AME = 0,0062; SH = 0,000681; z = 9,08; p < 0,001). Diğer değişkenler sabitken, 'anne_egitim' değişkeni 1 birim arttığında Geçme = 1 olma olasılığı ortalama 0,95 yüzde puan artmaktadır; bu etki %1 düzeyinde istatistiksel olarak anlamlıdır (AME = 0,0095; SH = 0,0030; z = 3,13; p = 0,002). 'burs' değişkeninde 'Evet' grubunun Geçme = 1 olma olasılığı, referans 


Tam rapor (Yöntem, tanımlayıcı istatistikler, katsayı ve marjinal etki tabloları, sınıflandırma tablosu, kaynakça): `logit.rapor("logit.docx")`
